In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("../data/raw/train_2.csv")
df

,new_id,Год,Месяц,Среднее количество промо товаров в чеке,Среднее количество товаров в чеке,Среднее количество отмен,Рабочие часы в день,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,...,"Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м),Количество касс,Флаг алкогольной лицензии,РТО
0,0,2024,1,1.08,6.03,147.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,7.514774e+07
1,0,2023,1,1.32,6.04,162.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,7.491475e+07
2,0,2025,1,0.82,6.00,145.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,8.712551e+07
3,0,2025,2,0.90,6.00,118.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,8.265980e+07
4,0,2024,2,1.25,6.06,154.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,7.420934e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485077,21745,2023,10,1.09,6.21,860.0,9.0,Средний по возрасту,Большой,Павловский Посад г,...,204,5,2,2,4,3,3,13,1,1.800939e+08
485078,21745,2024,11,0.91,6.21,1226.0,9.0,Средний по возрасту,Большой,Павловский Посад г,...,204,5,2,2,4,3,3,13,1,2.074941e+08
485079,21745,2023,11,1.16,6.36,755.0,9.0,Средний по возрасту,Большой,Павловский Посад г,...,204,5,2,2,4,3,3,13,1,1.792471e+08
485080,21745,2023,12,1.29,6.97,802.0,9.0,Средний по возрасту,Большой,Павловский Посад г,...,204,5,2,2,4,3,3,13,1,2.163958e+08


In [6]:
# Рабочие часы от 5 до 24
df_cleaned = df.copy()
df_cleaned['Рабочие часы в день'] = df_cleaned['Рабочие часы в день'].clip(5, 25)
df_cleaned['Рабочие часы в день'].value_counts()

Рабочие часы в день
14.0    65390
15.0    64012
13.0    57720
16.0    54600
12.0    47242
17.0    43264
11.0    34658
18.0    30134
10.0    21034
19.0    18694
9.0     12402
20.0    10556
8.0      6552
21.0     6084
22.0     3536
7.0      3094
25.0     1976
23.0     1508
6.0      1066
24.0      806
5.0       754
Name: count, dtype: int64

In [7]:
df_cleaned['Медицинские уч. и аптеки (300 м)'] = df_cleaned['Медицинские уч. и аптеки (300 м)'].clip(0, 10)
df_cleaned['Медицинские уч. и аптеки (300 м)'].value_counts()

Медицинские уч. и аптеки (300 м)
0     183638
1     105768
2      67782
3      45838
4      30108
5      19240
6      11960
7       7956
8       5044
10      4862
9       2886
Name: count, dtype: int64

In [8]:
df_cleaned['Останокви (300 м)'] = df_cleaned['Остановки (300 м)'].clip(0, 15)
df_cleaned['Останокви (300 м)'].value_counts()

Останокви (300 м)
0     267202
2      60996
3      38558
4      34034
1      32396
5      19370
6      12584
7       6396
8       4550
9       2730
10      2002
15      1482
11      1300
12       598
13       546
14       338
Name: count, dtype: int64

In [9]:
df_cleaned['Продуктовые магазины (500 м)'] = df_cleaned['Продуктовые магазины (500 м)'].clip(0, 15)
df_cleaned['Продуктовые магазины (500 м)'].value_counts()

Продуктовые магазины (500 м)
1     67340
2     62062
0     60424
3     59904
4     52676
5     45786
6     39078
7     30238
8     23556
9     16094
10    10920
11     6708
12     4550
13     2704
15     1560
14     1482
Name: count, dtype: int64

In [10]:
df_cleaned['Школы (300 м)'] = df_cleaned['Школы (300 м)'].clip(0, 5)
df_cleaned['Школы (300 м)'].value_counts()

Школы (300 м)
0    337740
1     98930
2     36192
3      8840
4      2444
5       936
Name: count, dtype: int64

In [11]:

df_encoded = df_cleaned.copy()

### Inflation adjustment to march 2025

In [14]:
# Load inflation coefficients for all months from 01(2023) to 03(2025)
# The inflation data for Russian Federation (ИПЦ) is at row 4 (0-indexed), column 2

start_year, start_month = 2023, 1
end_year, end_month = 2025, 3

sheet_names = []

# Generate months and read inflation data
for year in range(start_year, end_year + 1):
    months_range = 12 if year < end_year else end_month
    for month in range(1 if year > start_year else start_month, months_range + 1):
        sheet_name = f"{month:02d}({year})"
        sheet_names.append(sheet_name)

print(f"Total sheets to process: {len(sheet_names)}")
print(f"Sheets range: {sheet_names[0]} to {sheet_names[-1]}")

file_path = "../data/raw/ipc_RF_fo_sub_04-2026.xlsx"
inflation_monthly = {}
errors = []

for sheet_name in sheet_names:
    df_temp = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    # Увеличение цен продовольственных товаров к предыдущему месяцу
    inflation_monthly[sheet_name] = df_temp.iloc[4, 3]
    print(f"{sheet_name}: ИПЦ = {inflation_monthly[sheet_name]}")


print(f"\nSuccessfully loaded {len(inflation_monthly)} months")

Total sheets to process: 27
Sheets range: 01(2023) to 03(2025)
01(2023): ИПЦ = 101.32
02(2023): ИПЦ = 100.79
03(2023): ИПЦ = 100.13
04(2023): ИПЦ = 100.29
05(2023): ИПЦ = 99.69
06(2023): ИПЦ = 99.99
07(2023): ИПЦ = 100.49
08(2023): ИПЦ = 99.94
09(2023): ИПЦ = 100.86
10(2023): ИПЦ = 101.35
11(2023): ИПЦ = 101.55
12(2023): ИПЦ = 101.49
01(2024): ИПЦ = 101.26
02(2024): ИПЦ = 100.77
03(2024): ИПЦ = 100.17
04(2024): ИПЦ = 100.49
05(2024): ИПЦ = 100.41
06(2024): ИПЦ = 100.63
07(2024): ИПЦ = 100.36
08(2024): ИПЦ = 99.99
09(2024): ИПЦ = 100.34
10(2024): ИПЦ = 101.23
11(2024): ИПЦ = 102.33
12(2024): ИПЦ = 102.6
01(2025): ИПЦ = 101.33
02(2025): ИПЦ = 101.27
03(2025): ИПЦ = 100.83

Successfully loaded 27 months


In [16]:
# Calculate coefficients to adjust РТО to March 2025 prices
# The coefficient represents: price_march2025 = price_original * coefficient

# Sort sheet names chronologically (not alphabetically)
def parse_month_year(sheet_name):
    """Parse 'MM(YYYY)' format to (year, month) tuple"""
    month, year = sheet_name.split('(')
    year = int(year.rstrip(')'))
    month = int(month)
    return (year, month)

# Sort chronologically
sheet_names_chrono = sorted(inflation_monthly.keys(), key=parse_month_year)

inflation_coefficients = {}

for i, sheet_name in enumerate(sheet_names_chrono):
    # Start with 1.0 (no adjustment for March 2025 itself)
    coefficient = 1.0
    
    # Multiply by all inflation rates from the month AFTER current to March 2025
    # ИПЦ value represents the change from previous month to current month
    # So to go from current month to March 2025, multiply by all subsequent monthly factors
    for j in range(i + 1, len(sheet_names_chrono)):
        inflation_ipc = inflation_monthly[sheet_names_chrono[j]]
        # Convert ИПЦ (e.g., 100.84) to multiplier (1.0084)
        monthly_multiplier = inflation_ipc / 100.0
        coefficient *= monthly_multiplier
    
    inflation_coefficients[sheet_name] = coefficient

print("Inflation adjustment coefficients (multiply РТО by these to get March 2025 prices):\n")
print(f"{'Month':<12} {'Coefficient':<12}")
print("-" * 24)

for sheet_name in sheet_names_chrono:
    coeff = inflation_coefficients[sheet_name]
    print(f"{sheet_name:<12} {coeff:.8f}")

print(f"\nTotal coefficients calculated: {len(inflation_coefficients)}")
print(f"\nKey values:")
print(f"  01(2023) to 03(2025): {inflation_coefficients['01(2023)']:.8f}")
print(f"  12(2024) to 03(2025): {inflation_coefficients['12(2024)']:.8f}")
print(f"  03(2025) (no change): {inflation_coefficients['03(2025)']:.8f}")

Inflation adjustment coefficients (multiply РТО by these to get March 2025 prices):

Month        Coefficient 
------------------------
01(2023)     1.22673059
02(2023)     1.21711538
03(2023)     1.21553519
04(2023)     1.21202033
05(2023)     1.21578927
06(2023)     1.21591086
07(2023)     1.20998195
08(2023)     1.21070838
09(2023)     1.20038507
10(2023)     1.18439572
11(2023)     1.16631780
12(2023)     1.14919480
01(2024)     1.13489512
02(2024)     1.12622320
03(2024)     1.12431187
04(2024)     1.11882960
05(2024)     1.11426113
06(2024)     1.10728524
07(2024)     1.10331331
08(2024)     1.10342365
09(2024)     1.09968472
10(2024)     1.08632295
11(2024)     1.06158795
12(2024)     1.03468611
01(2025)     1.02110541
02(2025)     1.00830000
03(2025)     1.00000000

Total coefficients calculated: 27

Key values:
  01(2023) to 03(2025): 1.22673059
  12(2024) to 03(2025): 1.03468611
  03(2025) (no change): 1.00000000


In [17]:
# Create adjusted РТО column using inflation coefficients
# First, create a month-year identifier from Год and Месяц columns
df_adjusted = df_encoded.copy()
df_adjusted['month_year'] = df_adjusted['Месяц'].astype(str).str.zfill(2) + '(' + df_adjusted['Год'].astype(str) + ')'

# Apply coefficients to РТО
df_adjusted['РТО_adjusted'] = df_adjusted.apply(
    lambda row: row['РТО'] * inflation_coefficients.get(row['month_year'], 1.0), 
    axis=1
)

df_adjusted = df_adjusted.drop(columns=['month_year'])

print("РТО adjustment completed!")
print(f"\nSample comparison (original vs adjusted РТО):")
print(df_adjusted[['new_id', 'Год', 'Месяц', 'РТО', 'РТО_adjusted']].sort_values(['new_id', 'Год', 'Месяц']).reset_index(drop=True).head(26))

РТО adjustment completed!

Sample comparison (original vs adjusted РТО):
    new_id   Год  Месяц          РТО  РТО_adjusted
0        0  2023      1  74914754.22  9.190022e+07
1        0  2023      2  69240001.40  8.427307e+07
2        0  2023      3  79905726.38  9.712822e+07
3        0  2023      4  78567643.63  9.522558e+07
4        0  2023      5  80856174.96  9.830407e+07
5        0  2023      6  74635062.29  9.074958e+07
6        0  2023      7  73579921.54  8.903038e+07
7        0  2023      8  71929528.80  8.708568e+07
8        0  2023      9  69963064.37  8.398262e+07
9        0  2023     10  76474955.25  9.057661e+07
10       0  2023     11  75835109.27  8.844784e+07
11       0  2023     12  84825173.40  9.748065e+07
12       0  2024      1  75147744.85  8.528481e+07
13       0  2024      2  74209339.11  8.357628e+07
14       0  2024      3  78992997.13  8.881276e+07
15       0  2024      4  78183610.30  8.747414e+07
16       0  2024      5  83985250.56  9.358150e+07
17       

In [18]:
df_adjusted['РТО'] = df_adjusted['РТО_adjusted']
df_adjusted = df_adjusted.drop(columns=['РТО_adjusted'])

In [19]:
df_adjusted.to_csv("../data/processed/v1.csv", index=False)